# ARC-AGI-2 v6：作者 Qwen3-4B 格子模型流程（加固版）

演算法與 [sorokin/arc2-qwen3-unsloth-flash-lora-batch4-queue](https://www.kaggle.com/code/sorokin/arc2-qwen3-unsloth-flash-lora-batch4-queue) 相同：每題 rank-256 rsLoRA 測試時訓練（128 條增強序列）、16 視角 turbo-DFS、增強重評分。

v6 只加穩健性：路徑自動偵測、commit 跑 4 題冒煙／評分重跑跑全部題目、開始前先寫完整備援 submission.json、單題例外隔離、worker 互不連坐、原子寫檔、到期強制收尾。

In [ ]:
import time
global_start = time.time()

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile arc_v6_config.py
"""v6 runtime configuration: path discovery, rerun gating, task selection, deadlines.

No task IDs, hashes or counts of the public test file are pinned. Rerun mode takes every
task in the mounted test file; commit mode smoke-tests a few evaluation tasks chosen by
observed size only.
"""
import glob
import json
import os
import time

COMPETITION_SLUG = "arc-prize-2026-arc-agi-2"
MODEL_KEY = "qwen3_4b_grids15_sft139"
INPUT_ROOT = os.environ.get("ARC_V6_INPUT_ROOT", "/kaggle/input")
WORK_DIR = os.environ.get("ARC_V6_WORK_DIR", "/kaggle/working")
RERUN_WALL_SECONDS = 12 * 3600 - 600   # author global_end_time margin
COMMIT_WALL_SECONDS = 3600             # 4-task smoke; bounds the charged commit run
TASK_SECONDS = 1200                    # author per-task decode cap
TASK_FLOOR_SECONDS = 360               # v7: adaptive cap never below this (TTT + one decode batch)
COMMIT_TASK_SECONDS = 900              # v7: lower commit-smoke cap so the <TASK_SECONDS branch runs on hardware
SMOKE_TASKS = 4
NUM_WORKERS = 4
# v8: pass B re-runs tasks with independent seeds in the wall time pass A leaves unused; its candidates are
# pooled at assembly with pass A's top-1 always kept. Pass A itself is unchanged from v7.
PASS_B_MIN_SECONDS = float(os.environ.get("ARC_V8_PASS_B_MIN_SECONDS", "1200"))  # reload + floor + a task
# Pass B ends this much before end_time, so its kill + the two-store assembly stay well inside the 12 h limit.
PASS_B_DEADLINE_MARGIN = float(os.environ.get("ARC_V8_PASS_B_DEADLINE_MARGIN", "300"))
# v11: both passes run the luxluxshan half-cost width: 8 train augmentations x 8 geometries (64 TTT sequences)
# and 1 eval augmentation x 8 geometries (8 views), instead of the author's 16 / 2 (128 sequences, 16 views).
# Colab gate 2026-10-08 (colab-half-v1): half A took 0.555x full A; pooled half A+B pass@2 5.0 vs full A 3.0.
N_TRAIN_AUG = int(os.environ.get("ARC_V11_N_TRAIN_AUG", "8"))
N_EVAL_AUG = int(os.environ.get("ARC_V11_N_EVAL_AUG", "1"))
PASS_B_ENV = {"ARC_V8_TRAIN_AUG_SEED": "17", "ARC_V8_EVAL_AUG_SEED": "29", "ARC_V8_TRAINER_SEED": "137",
              "ARC_V8_LORA_SEED": "137", "ARC_V8_SCORE_SEED_OFFSET": "7"}
# v9: pass B first redraws the tasks pass A is unsure of that a floor cap would have cut, and only as many as
# their predicted full time fits (Colab pairing 2026-10-06: a second seed at cap 1200 solved a mid-size task pass A
# missed; at the 360 s floor it never did; tasks under ~360 s already got full redraws in v8).
PASS_B_CONFIDENT = float(os.environ.get("ARC_V9_PASS_B_CONFIDENT", "5"))  # kgmon top-1 score counted as sure
PASS_B_PLAN_SECONDS = float(os.environ.get("ARC_V9_PASS_B_PLAN_SECONDS", "120"))  # bound on reading store A
PASS_B_PLAN_FACTOR = 0.555  # v11: half-cost / full-cost seconds (Colab gate); v6-v8 Kaggle smokes ran at 0.5-1.0x the fit
PASS_B_START_STAGGER = 180  # pass-B ranks import one at a time (~44 s each) before taking tasks
PASS_B_PLAN_SCALE = float(os.environ.get("ARC_V9_PASS_B_PLAN_SCALE", "1"))  # test seam only; production = 1
SECONDS_FIT = (-223.08392554483123, 0.3000408772151657, -1.550904629974504e-05)  # colab-cal-v1 one-L4, cap 1200


def is_rerun():
    # Same truthiness as the author notebook.
    return bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))


def _single(paths, prefer, what):
    paths = sorted(set(paths))
    preferred = [p for p in paths if prefer in p.lower()]
    paths = preferred or paths
    if len(paths) != 1:
        raise RuntimeError(f"expected exactly one {what}, found {len(paths)}: {paths[:5]}")
    return paths[0]


def competition_dir(root=None):
    root = root or INPUT_ROOT
    hits = glob.glob(os.path.join(root, "**", "arc-agi_test_challenges.json"), recursive=True)
    return os.path.dirname(_single(hits, COMPETITION_SLUG, "competition test file"))


def model_dir(root=None):
    root = root or INPUT_ROOT
    hits = [os.path.dirname(p) for p in glob.glob(os.path.join(root, "**", "config.json"), recursive=True)
            if MODEL_KEY in p.lower()]
    return _single(hits, MODEL_KEY, "model directory")


def test_path(root=None):
    return os.path.join(competition_dir(root), "arc-agi_test_challenges.json")


def work_challenges_path(root=None):
    """Challenge file the model workers process in this mode."""
    name = "arc-agi_test_challenges.json" if is_rerun() else "arc-agi_evaluation_challenges.json"
    return os.path.join(competition_dir(root), name)


def _cells(task):
    return sum(len(g) * len(g[0]) for pairs in task.values() for pair in pairs for g in pair.values() if g)


def select_keys(challenges, rerun=None):
    """Rerun: every task, cheapest estimated work first (v7). Commit: size-chosen smoke set."""
    rerun = is_rerun() if rerun is None else rerun
    keys = sorted(challenges)
    if rerun:
        return sorted(keys, key=lambda k: (estimated_work(challenges[k]), k))
    by_size = sorted(keys, key=lambda k: (_cells(challenges[k]), k))
    chosen = [by_size[-1]]                                   # largest context: VRAM/length check
    multi = [k for k in by_size if len(challenges[k]["test"]) > 1 and k not in chosen]
    chosen += multi[:1]                                      # multiple test inputs
    chosen += [k for k in by_size if k not in chosen][:SMOKE_TASKS - len(chosen)]
    return sorted(chosen)


def estimated_work(task):
    """Size proxy: TTT cost grows with all grid cells, decoding with the number of test inputs."""
    try:
        return _cells(task) * (1 + 0.5 * (len(task["test"]) - 1))
    except Exception:  # an unexpected task shape must never stop the queue from being built
        return 0


def adaptive_cap(remaining_seconds, pending_tasks):
    """Fair share of the remaining wall over pending + in-flight tasks, within [floor, author cap]."""
    share = remaining_seconds * NUM_WORKERS / (max(0, pending_tasks) + NUM_WORKERS)
    upper = TASK_SECONDS if is_rerun() else COMMIT_TASK_SECONDS
    return max(TASK_FLOOR_SECONDS, min(upper, share))


def seed(name, default):
    """v8: per-pass seed; unset (pass A) keeps the v7 value."""
    return int(os.environ.get(name, default))


def outputs_dir():
    return os.environ.get("ARC_V6_OUTPUTS", "/kaggle/inference_outputs")


def outputs_dir_b():
    return os.environ.get("ARC_V8_OUTPUTS_B", outputs_dir().rstrip("/") + "_b")


def output_task(name):
    """Task key of a decoded-result file name ('<task>_<i>.<augmentation>')."""
    return name.split(".")[0].rsplit("_", 1)[0]


def covered_tasks(outputs):
    """Tasks with at least one finished decoded-result file in a store."""
    try:
        names = os.listdir(outputs)
    except OSError:
        names = []
    return {output_task(n) for n in names if not n.startswith("._tmp_")}


def failed_path(outputs):
    return outputs.rstrip("/") + "_failed.txt"


def mark_failed(key):
    """v8: record a task whose solve raised, so pass B tries it last; never raises."""
    try:
        with open(failed_path(outputs_dir()), "a") as f:
            f.write(json.dumps(key) + "\n")
    except Exception:
        pass


def failed_tasks(outputs):
    try:
        with open(failed_path(outputs)) as f:
            return {json.loads(line) for line in f if line.strip()}
    except Exception:
        return set()


def pass_b_keys(keys, outputs_a):
    """Pass B order (pass-A order kept within groups): tasks pass A left without any decoded result and that
    did not raise, then tasks with results, then tasks that raised without results (likely to fail again)."""
    covered, failed = covered_tasks(outputs_a), failed_tasks(outputs_a)
    missing = [k for k in keys if k not in covered and k not in failed]
    return missing + [k for k in keys if k in covered] + [k for k in keys if k not in covered and k in failed]


def task_confidence(keys, outputs_a, challenges, deadline):
    """v9: per task, the minimum over its test outputs of pass A's kgmon top-1 score; an output without any
    readable candidate counts as -inf (unsure). Raises TimeoutError past the deadline."""
    import bz2
    import pickle
    from arc_decoder import getter_kgmon, hashable
    groups = {}
    for name in sorted(os.listdir(outputs_a)):
        if time.time() > deadline:
            raise TimeoutError("reading pass-A store took too long")
        if name.startswith("._tmp_"):
            continue
        try:
            with bz2.BZ2File(os.path.join(outputs_a, name)) as f:
                samples = pickle.load(f)
        except Exception:
            continue  # same tolerance as the assembly loader
        by_grid = groups.setdefault(name.split(".")[0], {})
        for sample in samples:
            try:
                by_grid.setdefault(hashable(sample["solution"]), []).append(sample)
            except Exception:
                pass
    conf = {}
    for k in keys:
        scores = []
        for i in range(len(challenges[k]["test"])):
            best = float("-inf")
            for guesses in groups.get(f"{k}_{i}", {}).values():
                try:
                    best = max(best, float(getter_kgmon(guesses)))
                except Exception:
                    pass
            scores.append(best)
        conf[k] = min(scores) if scores else float("-inf")
    return conf


def task_tokens(task):
    """Prompt length the model sees (colab-cal-v1 analyze.tokens): train pairs + the largest test input + 900."""
    gl = lambda g: len(g) * (len(g[0]) + 1)
    return min(8192, sum(gl(p["input"]) + gl(p["output"]) + 10 for p in task["train"])
               + max(gl(q["input"]) for q in task["test"]) + 900)


def full_seconds(task):
    """v9: predicted Kaggle seconds for one uncut solve (TTT + decode); an odd shape plans the author cap."""
    try:
        L = task_tokens(task)
        return (SECONDS_FIT[0] + SECONDS_FIT[1] * L + SECONDS_FIT[2] * L * L) * PASS_B_PLAN_FACTOR
    except Exception:
        return float(TASK_SECONDS)


def pass_b_plan(keys, outputs_a, challenges, seconds_left):
    """v9 pass-B queue, pass-A cheap-first order kept within every group: tasks pass A left without results; tasks
    pass A is unsure of whose full solve a floor cap would cut; the other unsure tasks (v8 already redrew those
    fully); sure tasks; tasks that raised. Truncated where the planned seconds (full solve, clipped to the
    [floor, author] caps) exceed the pass-B worker budget. Any error falls back to v8's full queue (pass_b_keys).
    Returns (keys, info)."""
    start = time.time()
    try:
        conf = task_confidence(keys, outputs_a, challenges, start + PASS_B_PLAN_SECONDS)
        covered, failed = covered_tasks(outputs_a), failed_tasks(outputs_a)
        full = {k: full_seconds(challenges[k]) for k in keys}
        missing = [k for k in keys if k not in covered and k not in failed]
        unsure = [k for k in keys if k in covered and not conf[k] >= PASS_B_CONFIDENT]
        long_ = [k for k in unsure if full[k] > TASK_FLOOR_SECONDS]
        short = [k for k in unsure if not full[k] > TASK_FLOOR_SECONDS]
        sure = [k for k in keys if k in covered and conf[k] >= PASS_B_CONFIDENT]
        raised = [k for k in keys if k not in covered and k in failed]
        budget = (seconds_left - (time.time() - start) - PASS_B_START_STAGGER * PASS_B_PLAN_SCALE) * NUM_WORKERS
        ordered, planned, total = missing + long_ + short + sure + raised, [], 0.0
        for k in ordered:
            need = min(TASK_SECONDS, max(TASK_FLOOR_SECONDS, full[k])) * PASS_B_PLAN_SCALE
            if len(planned) >= NUM_WORKERS and total + need > budget:
                break
            planned.append(k)
            total += need
        info = {"pass_b_order": "confidence", "missing": len(missing), "unsure_long": len(long_),
                "unsure_short": len(short), "sure": len(sure), "raised": len(raised), "limit": len(planned),
                "planned_sum": round(total), "budget": round(budget), "threshold": PASS_B_CONFIDENT,
                "plan_seconds": round(time.time() - start, 1)}
        return planned, info
    except Exception as exc:
        return pass_b_keys(keys, outputs_a), {"pass_b_order": "fallback", "error": f"{type(exc).__name__}: {exc}"[:300],
                                              "plan_seconds": round(time.time() - start, 1)}


def decode_batches(test_id_to_subkeys, length_of, batch_size=4):
    """Decode batches per test input. With the author's 16 views (8 geometries x 2 permutations, sorted subkeys)
    the author's fixed pairing is kept verbatim; any other view count (v11 half-cost: 8 views) batches prompts of
    equal token length (batched DFS needs equal lengths), at most batch_size each, in sorted subkey order."""
    if all(len(subkeys) == 16 for subkeys in test_id_to_subkeys.values()):
        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            for offsets in ([0, 4], [2, 6]):
                batches.append([sk for offset in offsets for sk in subkeys[offset:offset + 2]])
        for test_id, subkeys in test_id_to_subkeys.items():
            for offsets in ([8, 12], [10, 14]):
                batches.append([sk for offset in offsets for sk in subkeys[offset:offset + 2]])
        return batches
    batches = []
    for test_id, subkeys in test_id_to_subkeys.items():
        groups = {}
        for subkey in subkeys:
            groups.setdefault(length_of(subkey), []).append(subkey)
        for group in groups.values():
            batches.extend(group[i:i + batch_size] for i in range(0, len(group), batch_size))
    return batches


def end_time(start):
    return start + (RERUN_WALL_SECONDS if is_rerun() else COMMIT_WALL_SECONDS)


def atomic_json(path, value):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(value, f)
    os.replace(tmp, path)


def fallback_submission(test_challenges):
    return {k: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in task["test"]]
            for k, task in test_challenges.items()}


def validate_submission(test_challenges, submission):
    if set(submission) != set(test_challenges):
        raise ValueError("submission keys differ from test keys")
    for k, task in test_challenges.items():
        entries = submission[k]
        if not isinstance(entries, list) or len(entries) != len(task["test"]):
            raise ValueError(f"{k}: wrong number of test entries")
        for e in entries:
            if not isinstance(e, dict) or set(e) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"{k}: attempts missing")
            if not all(valid_grid(g) for g in e.values()):
                raise ValueError(f"{k}: invalid grid")
    return True


def valid_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and isinstance(g[0], list) and 1 <= len(g[0]) <= 30
            and all(isinstance(r, list) and len(r) == len(g[0]) for r in g)
            and all(type(c) is int and 0 <= c <= 9 for r in g for c in r))


def log(message, **fields):
    print(json.dumps({"t": round(time.time(), 1), "msg": message, **fields}), flush=True)


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
try:
    from transformers import AutoTokenizer
except ImportError:  # v6: CPU hidden-shape tests run without transformers
    AutoTokenizer = None


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.rsplit('_', 1)
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in sorted(os.listdir(store)):
            if key.startswith("._tmp_"):
                continue  # v6: unfinished atomic write
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as exc:  # v6: a truncated file must not drop the submission
                print(f"*** skip unreadable decoded result {key}: {type(exc).__name__}")
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
import arc_v6_config

import gc
import traceback
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=arc_v6_config.seed("ARC_V8_LORA_SEED", 42),  # v8: pass B reseeds
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=arc_v6_config.seed("ARC_V8_TRAINER_SEED", 42),
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=arc_v6_config.model_dir(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    test_path = arc_v6_config.work_challenges_path()

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = arc_v6_config.outputs_dir()  # v8: pass B writes to its own store
    os.makedirs(dir_outputs, exist_ok=True)

    consecutive_failures = 0

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        # v7: a task picked up in the last TASK_FLOOR_SECONDS cannot finish TTT + one batch.
        if end_time - time.time() < arc_v6_config.TASK_FLOOR_SECONDS:
            print(f"[Rank {rank}] stop: {end_time - time.time():.0f}s left, below task floor", flush=True)
            break

        # v7: shrink the per-task cap when the remaining wall cannot cover the queue at 1200 s/task.
        # qsize still holds the NUM_WORKERS None sentinels while real tasks remain.
        task_cap = arc_v6_config.adaptive_cap(end_time - time.time(), queue.qsize() - arc_v6_config.NUM_WORKERS)
        print(f"[Rank {rank}] task {key} cap {task_cap:.0f}s", flush=True)

        failed = False
        try:
            model = solve_task(key, model, tokenizer, formatter, collator, default_weights, train_args,
                               arc_test_set, max_seq_length, max_new_tokens, max_score, dir_outputs, rank, end_time,
                               task_cap)
            consecutive_failures = 0
        except Exception as exc:  # v6: one failing task must not stop this GPU's queue
            failed = True
            # v7: an OOM on one large task is usually recoverable, so it weighs half (6 in a row retire the worker).
            consecutive_failures += 0.5 if isinstance(exc, torch.cuda.OutOfMemoryError) else 1
            print(f"[Rank {rank}] task {key} failed: {type(exc).__name__}: {str(exc)[:300]}", flush=True)
            arc_v6_config.mark_failed(key)  # v8: pass B tries tasks that raised last
            traceback.print_exc()
            # A sticky CUDA fault would make this worker drain the shared queue; leave it to the others.
            if "CUDA error" in str(exc) or consecutive_failures >= 3:
                print(f"[Rank {rank}] stopping after {consecutive_failures} failure(s)", flush=True)
                break
        if failed:  # v7: clean up after the except block releases the traceback (trainer/optimizer refs)
            gc.collect()
            torch.cuda.empty_cache()


def solve_task(key, model, tokenizer, formatter, collator, default_weights, train_args,
               arc_test_set, max_seq_length, max_new_tokens, max_score, dir_outputs, rank, end_time, task_cap=arc_v6_config.TASK_SECONDS):
        start_time = time.time()
        decoded_test_ids = set()
        score_seed_offset = arc_v6_config.seed("ARC_V8_SCORE_SEED_OFFSET", 0)
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=arc_v6_config.N_TRAIN_AUG, shfl_keys=True, seed=arc_v6_config.seed("ARC_V8_TRAIN_AUG_SEED", 1))
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=arc_v6_config.N_EVAL_AUG, seed=arc_v6_config.seed("ARC_V8_EVAL_AUG_SEED", 2))
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].rsplit("_", 1)[1]
            test_id_to_subkeys[test_id].append(subkey)

        # v11: batching moved to arc_v6_config.decode_batches (author pairing kept verbatim for 16 views;
        # other view counts, e.g. half-cost 8 views, batch equal-length prompts).
        batches = arc_v6_config.decode_batches(
            test_id_to_subkeys, lambda sk: len(tokenizer.encode(eval_ds.get(sk, formatter)["input"])))

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                batch_test_id = subkeys[0].split(".")[0].rsplit("_", 1)[1]
                # v7: every test input gets at least one decode batch after (unbounded) TTT;
                # further batches respect the (adaptive) per-task cap.
                first_for_test = batch_test_id not in decoded_test_ids
                if time.time() > end_time or (spend_time > task_cap and not first_for_test):
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    if time.time() > end_time:
                        break
                    continue
                decoded_test_ids.add(batch_test_id)
                # Author behaviour at the full cap; below it, DFS stops at the task deadline but
                # always keeps a short window for a guaranteed first batch.
                dfs_end_time = end_time if task_cap >= arc_v6_config.TASK_SECONDS else min(
                    end_time, max(start_time + task_cap, time.time() + 180))

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, dfs_end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=(hash(bk) + score_seed_offset) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        tmp_path = os.path.join(dir_outputs, "._tmp_" + subkey)
                        with bz2.BZ2File(tmp_path, "w") as f:
                            pickle.dump(decoded_result, f)
                        os.replace(tmp_path, os.path.join(dir_outputs, subkey))  # v6: atomic

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")
        return model


In [ ]:
%%writefile starter.py
import argparse
import importlib
import json
import multiprocessing as mp
import os
import time

import arc_v6_config as cfg

MARKER_DIR = os.environ.get("ARC_V6_MARKER_DIR", "/kaggle")
SOLVER = os.environ.get("ARC_V6_SOLVER_MODULE", "arc_solver")  # test seam; production = arc_solver
MARKER_WAIT_SECONDS = 900
MARKER_POLL_SECONDS = float(os.environ.get("ARC_V6_MARKER_POLL", "5"))
KILL_GRACE_SECONDS = float(os.environ.get("ARC_V6_KILL_GRACE", "120"))


def marker(rank, label="A"):
    return os.path.join(MARKER_DIR, f"worker{rank}" + ("" if label == "A" else "." + label))


def local_worker(rank, queue, end_time, label="A", env=None):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ.update(env or {})  # v8: pass B seeds and output store, set before the solver is imported

    if SOLVER == "arc_solver":
        import torch
        torch.set_default_device("cpu")

    # Fix Unsloth patching issue (author): import ranks one at a time, but never wait forever.
    if rank > 0:
        waited = 0
        while not os.path.exists(marker(rank - 1, label)) and waited < MARKER_WAIT_SECONDS:
            time.sleep(MARKER_POLL_SECONDS)
            waited += MARKER_POLL_SECONDS

    worker = importlib.import_module(SOLVER).worker

    with open(marker(rank, label), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!" + ("" if label == "A" else f" pass {label}"), flush=True)

    worker(rank, queue, end_time)

    print(f"[Rank {rank}] done!", flush=True)


def run_pass(ctx, manager, keys, end_time, label="A", env=None):
    """One pass: NUM_WORKERS independent GPU processes over a shared queue. True when every worker is gone."""
    queue = manager.Queue()
    for key in keys:
        queue.put(key)
    for _ in range(cfg.NUM_WORKERS):
        queue.put(None)

    # Independent processes: one crashed GPU worker must not kill the other three
    # (torch.multiprocessing.spawn terminates all ranks when one fails).
    procs = [ctx.Process(target=local_worker, args=(rank, queue, end_time, label, env)) for rank in range(cfg.NUM_WORKERS)]
    for p in procs:
        p.start()
    while any(p.is_alive() for p in procs) and time.time() < end_time + KILL_GRACE_SECONDS:
        time.sleep(2)
    for p in procs:
        if p.is_alive():
            p.terminate()
    for p in procs:
        p.join(15)
        if p.is_alive():
            p.kill()
            p.join(5)
    cfg.log("workers_exited", exitcodes=[p.exitcode for p in procs], pass_label=label)
    return not any(p.is_alive() for p in procs)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    with open(cfg.work_challenges_path(), "r") as f:
        data = json.load(f)
    keys = cfg.select_keys(data)
    cfg.log("queue", rerun=cfg.is_rerun(), tasks=len(keys), challenges=len(data),
            seconds_left=round(args.end_time - time.time()))

    # v7: fixed hash seed so hash(bk)-seeded augmentation scoring is reproducible across runs.
    os.environ.setdefault("PYTHONHASHSEED", "0")
    ctx = mp.get_context("spawn")
    manager = ctx.Manager()

    clean = run_pass(ctx, manager, keys, args.end_time)

    # v8: pass B only in wall time pass A did not need; a saturated run behaves exactly like v7.
    end_b = args.end_time - cfg.PASS_B_DEADLINE_MARGIN
    seconds_left = end_b - time.time()
    if clean and seconds_left >= cfg.PASS_B_MIN_SECONDS:
        keys_b, plan = cfg.pass_b_plan(keys, cfg.outputs_dir(), data, seconds_left)  # v9: unsure tasks, near-full caps
        env_b = dict(cfg.PASS_B_ENV, ARC_V6_OUTPUTS=cfg.outputs_dir_b())
        missing = len(set(keys) - cfg.covered_tasks(cfg.outputs_dir()))
        cfg.log("pass_b_start", tasks=len(keys_b), missing_after_a=missing,
                failed_after_a=len(cfg.failed_tasks(cfg.outputs_dir())), seconds_left=round(end_b - time.time()), **plan)
        clean = run_pass(ctx, manager, keys_b, end_b, label="B", env=env_b)
    else:
        cfg.log("pass_b_skipped", clean=clean, seconds_left=round(seconds_left))
    if not clean:
        # An unkillable child would block multiprocessing's atexit join forever.
        cfg.log("abandoning_unkillable_workers")
        os._exit(0)


In [ ]:
%%writefile assemble.py
"""Final cell: merge decoded results into a full-coverage submission.json."""
import json
import os
import traceback

import numpy as np

import arc_v6_config as cfg
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

OUTPUTS = os.environ.get("ARC_V6_OUTPUTS", "/kaggle/inference_outputs")


def load_decoder(dataset, stores):
    decoder = ArcDecoder(dataset, n_guesses=2)
    for store, run_name in stores:
        if os.path.isdir(store):
            decoder.load_decoded_results(store, run_name=run_name)
    known = set(dataset.keys)
    stale = [k for k in decoder.decoded_results if k not in known]
    for k in stale:  # never let an unknown key break the merge
        del decoder.decoded_results[k]
    return decoder


def keep_a_top1(sel_a, sel_p):
    """v8: pass A's top-1 is always one of the two attempts; the other is the best distinct pooled grid.

    With no pass-B candidates for an output, the pooled ranking equals pass A's, so this is v7's choice.
    """
    selected, changed = {}, 0
    for bk in set(sel_a) | set(sel_p):
        ranked_a, ranked_p = list(sel_a.get(bk, [])), list(sel_p.get(bk, []))
        if not ranked_a:
            selected[bk] = ranked_p[:2]
        else:
            first = ranked_a[0]
            other = next((g for g in ranked_p + ranked_a[1:] if not np.array_equal(g, first)), None)
            selected[bk] = [first] + ([other] if other is not None else [])
        before = ranked_a[:2]
        changed += len(before) != len(selected[bk]) or any(
            not np.array_equal(x, y) for x, y in zip(before, selected[bk]))
    return selected, changed


def model_submission(challenges_path, outputs_dir, solutions_path=None, use_b=True):
    data = ArcDataset.from_file(challenges_path)
    if solutions_path:
        data = data.load_replies(solutions_path)
    multi = data.split_multi_replies()
    outputs_b = cfg.outputs_dir_b()
    decoder_a = load_decoder(multi, [(outputs_dir, "")])
    pooled = load_decoder(multi, [(outputs_dir, ""), (outputs_b, ".b")]) if use_b else decoder_a
    selected, changed = keep_a_top1(decoder_a.run_selection_algo(), pooled.run_selection_algo())
    stats = {"outputs_a": len(decoder_a.decoded_results), "outputs_pooled": len(pooled.decoded_results),
             "pass_b_files": len(os.listdir(outputs_b)) if use_b and os.path.isdir(outputs_b) else 0,
             "changed_vs_pass_a": int(changed)}
    return data, decoder_a, pooled, data.get_submission(selected), stats


def merge(test, fallback, candidate):
    """Per attempt: take the model grid if valid, else keep the fallback grid."""
    merged, used = {}, 0
    for key, task in test.items():
        entries = []
        for i in range(len(task["test"])):
            entry = dict(fallback[key][i])
            for attempt in ("attempt_1", "attempt_2"):
                try:
                    grid = np.asarray(candidate[key][i][attempt]).tolist()
                except Exception:
                    continue
                if cfg.valid_grid(grid):
                    entry[attempt] = grid
                    used += grid != [[0]]
            entries.append(entry)
        merged[key] = entries
    return merged, used


def main(work_dir=None, outputs_dir=OUTPUTS):
    work_dir = work_dir or cfg.WORK_DIR
    test_path = cfg.test_path()
    with open(test_path) as f:
        test = json.load(f)
    fallback = cfg.fallback_submission(test)
    final, used, error, pool_stats = fallback, 0, None, {}
    if cfg.is_rerun():
        try:
            try:
                _, _, _, candidate, pool_stats = model_submission(test_path, outputs_dir)
            except Exception:  # v8: a pass-B problem must never cost pass A's answers
                pool_stats = {"pooling_error": traceback.format_exc()[-1500:]}
                cfg.log("pooling_failed_using_pass_a")
                _, _, _, candidate, stats_a = model_submission(test_path, outputs_dir, use_b=False)
                pool_stats.update(stats_a)
            final, used = merge(test, fallback, candidate)
            cfg.validate_submission(test, final)
        except Exception:
            error = traceback.format_exc()
            final, used = fallback, 0
    else:
        # Commit smoke: model ran on evaluation tasks; report them separately, keep the
        # test-file submission full-coverage fallback so the version stays submittable.
        comp = cfg.competition_dir()
        sol = os.path.join(comp, "arc-agi_evaluation_solutions.json")
        try:
            data, decoder, pooled, candidate, pool_stats = model_submission(
                cfg.work_challenges_path(), outputs_dir, sol if os.path.exists(sol) else None)
            cfg.atomic_json(os.path.join(work_dir, "eval-smoke-submission.json"), candidate)
            if pooled.decoded_results and os.path.exists(sol):
                for label, dec in (("pass_a", decoder), ("pooled", pooled)):
                    try:
                        print(f"*** {label}")
                        dec.benchmark_selection_algos()
                    except Exception as exc:  # author benchmark fails when nothing is solved
                        cfg.log("benchmark_skipped", label=label, error=type(exc).__name__)
                ran = sorted({b.rsplit("_", 1)[0] for b in pooled.decoded_results})
                sel_a = decoder.run_selection_algo()
                solved_a = sum(np.mean([any(np.array_equal(r, g) for g in sel_a.get(f"{k}_{i}", [])[:2])
                                        for i, r in enumerate(data.replies[k])]) for k in ran)
                solved = sum(np.mean([any(np.array_equal(r, e[a]) for a in ("attempt_1", "attempt_2"))
                                      for r, e in zip(data.replies[k], candidate[k])]) for k in ran)
                cfg.log("eval_smoke_score", solved=float(solved), tasks=len(ran), solved_pass_a=float(solved_a),
                        contaminated_by_pretraining=True)
        except Exception:
            error = traceback.format_exc()
    cfg.validate_submission(test, final)
    cfg.atomic_json(os.path.join(work_dir, "submission.json"), final)
    record = {"rerun": cfg.is_rerun(), "test_tasks": len(test),
              "test_inputs": sum(len(t["test"]) for t in test.values()),
              "model_attempts_used": used, "error": error,
              "decoded_files": len(os.listdir(outputs_dir)) if os.path.isdir(outputs_dir) else 0, **pool_stats}
    cfg.atomic_json(os.path.join(work_dir, "v6-assembly.json"), record)
    cfg.log("assembled", **{k: v for k, v in record.items() if k != "error"}, error=bool(error))
    if error:
        print(error[-3000:])
    return record


if __name__ == "__main__":
    main()


In [ ]:
import json
import arc_v6_config as cfg
global_end_time = cfg.end_time(global_start)
with open(cfg.test_path()) as f:
    _test = json.load(f)
cfg.atomic_json(cfg.WORK_DIR + '/submission.json', cfg.fallback_submission(_test))
cfg.log('fallback_written', rerun=cfg.is_rerun(), test_tasks=len(_test), model_dir=cfg.model_dir(),
        competition_dir=cfg.competition_dir(),
        work_file=cfg.work_challenges_path(), seconds_left=round(global_end_time - time.time()))

In [ ]:
!PYTHONUNBUFFERED=1 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import assemble
record = assemble.main()